# Monitor 02 — modelos locales por serie + baselines comunes

Compara exclusivamente los cuatro modelos locales. Los baselines son idénticos a los del monitor Financial-GPT: `NAIVE_LAST_VALUE`, `NAIVE_ZERO` sólo para series de variación y `SEASONAL_NAIVE_FINANCIAL` con ciclo semanal configurable.

In [ ]:
# Parámetros
AUTO_DISCOVER_LOCAL = True

LOCAL_EXECUTION_ROOT = (
    "s3://your-private-bucket/"
    "data/sandboxes/m6hn/data/coe_liquidez_diaria/execution"
)

# Úsalos sólo cuando quieras fijar runs concretos.
LOCAL_RUN_URIS = {
    # "MLP_E_D": "s3://.../MLP_E_D_...",
    # "MLP_VaE_D": "s3://.../MLP_VaE_D_...",
    # "RNN_E_D": "s3://.../RNN_E_D_...",
    # "RNNBi_E_D": "s3://.../RNNBi_E_D_...",
}

WINNER_METRICS = ["MAE", "RMSE", "WMAPE", "EVS", "R2"]
INCLUDE_BASELINES = True
SEASONAL_PERIOD_DAYS = 7
OUTPUT_DIRECTORY = "./local_models_monitor_output"
UPLOAD_TO_S3 = True  # conserva el export histórico bajo execution/<fecha>/ensemble/

In [ ]:
from IPython.display import display

from financial_gpt_monitor import (
    compare_local_financial_runs,
    discover_latest_local_runs,
)

## 1. Resolver los cuatro runs locales

In [ ]:
local_runs = (
    discover_latest_local_runs(LOCAL_EXECUTION_ROOT)
    if AUTO_DISCOVER_LOCAL
    else dict(LOCAL_RUN_URIS)
)

print("Runs locales:")
for name, uri in local_runs.items():
    print(" ", name, "->", uri)

## 2. Comparación común y ganador local/baseline por serie

In [ ]:
monitor = compare_local_financial_runs(
    local_run_uris=local_runs,
    metrics=WINNER_METRICS,
    include_baselines=INCLUDE_BASELINES,
    seasonal_period_days=SEASONAL_PERIOD_DAYS,
)

display(monitor.run_inventory)
display(monitor.comparison_coverage)
display(monitor.winner_counts)
display(monitor.winners_by_series)
display(monitor.ensemble_forecast.head(50))

## 3. Contrato legacy `final` y `final_long`

In [ ]:
final_long = (
    monitor.ensemble_forecast
    .select(["date", "cross_key_id", "pred_orig"])
    .rename({"cross_key_id": "serie"})
    .to_pandas()
)
final = (
    final_long
    .pivot(index="date", columns="serie", values="pred_orig")
    .sort_index()
)

display(final)
display(final_long.head(50))

## 4. Guardar salidas y gates

In [ ]:
output = monitor.write(OUTPUT_DIRECTORY)

assert monitor.run_inventory.height == 7
assert monitor.winners_by_series.height > 0
assert monitor.ensemble_forecast.height > 0
assert set(monitor.run_inventory["family"].unique()).issubset({"local", "baseline"})
assert set(monitor.winners_by_series["winner_family"].unique()).issubset({"local", "baseline"})
assert "global" not in monitor.run_inventory["family"].unique().to_list()

# Conserva el comportamiento histórico: publicar las salidas en la carpeta
# ensemble/ de la misma fecha de ejecución cuando los runs provienen de S3.
if UPLOAD_TO_S3 and all(str(uri).startswith("s3://") for uri in local_runs.values()):
    from pathlib import Path
    from urllib.parse import urlparse
    import boto3

    parents = {str(uri).rstrip("/").rsplit("/", 1)[0] for uri in local_runs.values()}
    if len(parents) != 1:
        raise ValueError(f"Los runs locales no pertenecen a la misma fecha: {sorted(parents)}")
    ensemble_uri = next(iter(parents)) + "/ensemble"
    parsed = urlparse(ensemble_uri)
    client = boto3.client("s3")
    prefix = parsed.path.lstrip("/").rstrip("/")
    for artifact in Path(output).iterdir():
        if artifact.is_file():
            client.upload_file(str(artifact), parsed.netloc, f"{prefix}/{artifact.name}")
    client.put_object(
        Bucket=parsed.netloc,
        Key=f"{prefix}/forecast_timeseries_FINAL_wide.csv",
        Body=final.to_csv().encode("utf-8"),
    )
    client.put_object(
        Bucket=parsed.netloc,
        Key=f"{prefix}/forecast_timeseries_FINAL_long.csv",
        Body=final_long.to_csv(index=False).encode("utf-8"),
    )
    print("Salidas S3:", ensemble_uri)

print("Artefactos locales:", output)
print("Candidatos: 4 locales + 3 baselines")
print("Baselines: NAIVE_LAST_VALUE, NAIVE_ZERO para variación, SEASONAL_NAIVE_FINANCIAL")
print("Periodo estacional:", SEASONAL_PERIOD_DAYS, "días")
print("Series comparadas:", monitor.winners_by_series.height)
print("Filas del ensemble:", monitor.ensemble_forecast.height)
print("✅ ningún modelo global entra a este monitor")
print("✅ métricas recalculadas sobre fechas comunes de test")